# Scriptorium — Barton HTR Training (Kaggle)

Fine-tunes a Kraken HTR model on 3,148 labelled row crops from the 1850 U.S. Census (Barton, Tioga Co., NY).

**Setup:** sidebar → Accelerator → **GPU P100** (or T4 x2). Then **Save Version → Save & Run All (Commit)** — the notebook runs on Kaggle's servers, no browser tab required.

The trained model is written to `/kaggle/working/barton_htr.mlmodel` and appears in the notebook's Output tab after the commit finishes.

In [ ]:
# 1. Install kraken (git-lfs is preinstalled on Kaggle).
!pip install -q kraken
!kraken --version
!git lfs --version

In [ ]:
# 2. Clone the repo (LFS pointer for the tarball, then explicit pull).
%cd /kaggle/working
!rm -rf scriptorium-rails
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/kraftinator/scriptorium-rails.git
%cd scriptorium-rails
!git lfs pull -I data/barton_htr_data.tar.gz
!ls -lh data/barton_htr_data.tar.gz

In [ ]:
# 3. Extract training data and emit .gt.txt companions.
%cd /kaggle/working/scriptorium-rails
!mkdir -p data-extract
!tar xzf data/barton_htr_data.tar.gz -C data-extract
# Rewrite manifest paths to point at the extracted location.
!sed -i 's|/Users/admin/scriptorium/data|/kaggle/working/scriptorium-rails/data-extract|g' data-extract/manifest.jsonl
!python python/prep_kraken_gt.py data-extract/manifest.jsonl
# Build the training image list.
!find data-extract/crops -name 'line_*.png' | while read p; do [ -f "${p%.png}.gt.txt" ] && echo "$p"; done > data-extract/train_images.lst
!wc -l data-extract/train_images.lst

In [ ]:
# 4. Verify GPU.
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 5. Train. Batch 32 on GPU, early stopping (patience 5).
!mkdir -p /kaggle/working/models
!ketos -v train \
    -B 32 \
    -o /kaggle/working/models/barton_htr \
    -q early \
    --lag 5 \
    -F 1.0 \
    -d cuda:0 \
    -f path \
    --workers 4 \
    -t data-extract/train_images.lst 2>&1 | tail -150

In [ ]:
# 6. Copy the latest checkpoint into /kaggle/working root so it shows up
#    in the notebook's Output tab after commit.
!ls -lh /kaggle/working/models/
!cp "$(ls -t /kaggle/working/models/barton_htr_*.mlmodel | head -1)" /kaggle/working/barton_htr.mlmodel
!ls -lh /kaggle/working/barton_htr.mlmodel